In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv(r"C:\Users\rjmax\Desktop\mol-property-pred\tox21.csv")

targets = [c for c in df.columns if c not in ["mol_id", "smiles"]]
print("Target coverage:\n")
for t in targets:
    total = df[t].notna().sum()
    toxic = df[t].sum()
    print(f"{t:<20} tested: {total:>4} | toxic: {int(toxic):>3} | toxic%: {toxic/total*100:.1f}%")
    
print(f"Total molecules: {len(df)}")

Target coverage:

NR-AR                tested: 7265 | toxic: 309 | toxic%: 4.3%
NR-AR-LBD            tested: 6758 | toxic: 237 | toxic%: 3.5%
NR-AhR               tested: 6549 | toxic: 768 | toxic%: 11.7%
NR-Aromatase         tested: 5821 | toxic: 300 | toxic%: 5.2%
NR-ER                tested: 6193 | toxic: 793 | toxic%: 12.8%
NR-ER-LBD            tested: 6955 | toxic: 350 | toxic%: 5.0%
NR-PPAR-gamma        tested: 6450 | toxic: 186 | toxic%: 2.9%
SR-ARE               tested: 5832 | toxic: 942 | toxic%: 16.2%
SR-ATAD5             tested: 7072 | toxic: 264 | toxic%: 3.7%
SR-HSE               tested: 6467 | toxic: 372 | toxic%: 5.8%
SR-MMP               tested: 5810 | toxic: 918 | toxic%: 15.8%
SR-p53               tested: 6774 | toxic: 423 | toxic%: 6.2%
Total molecules: 7831


In [3]:
def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing()),
        atom.GetTotalNumHs(),
    ]

def smiles_to_graph_multitask(smiles, labels):
    """
    labels: list of 12 values, each is 0.0, 1.0, or NaN
    Returns a PyG Data object with:
      - x: atom features
      - edge_index: bond connectivity  
      - y: 12 toxicity labels (NaN replaced with -1 as mask signal)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    node_feats = [atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)
    
    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
    
    if len(edges) == 0:
        return None
    
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    
    # Replace NaN with -1 so we can mask them during training
    clean_labels = [-1.0 if np.isnan(l) else float(l) for l in labels]
    y = torch.tensor(clean_labels, dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, y=y)

# Build dataset
print("Converting molecules to graphs...")
dataset = []
for _, row in df.iterrows():
    smiles = row["smiles"]
    labels = [row[t] for t in targets]
    graph = smiles_to_graph_multitask(smiles, labels)
    if graph is not None:
        dataset.append(graph)

print(f"Successfully converted: {len(dataset)} molecules")

Converting molecules to graphs...


[15:20:37] WARNING: not removing hydrogen atom without neighbors
[15:20:37] Explicit valence for atom # 8 Al, 6, is greater than permitted
[15:20:37] Explicit valence for atom # 3 Al, 6, is greater than permitted
[15:20:37] Explicit valence for atom # 4 Al, 6, is greater than permitted
[15:20:38] Explicit valence for atom # 4 Al, 6, is greater than permitted
[15:20:38] Explicit valence for atom # 9 Al, 6, is greater than permitted
[15:20:38] Explicit valence for atom # 5 Al, 6, is greater than permitted
[15:20:38] Explicit valence for atom # 16 Al, 6, is greater than permitted
[15:20:39] Explicit valence for atom # 20 Al, 6, is greater than permitted


Successfully converted: 7804 molecules


In [4]:
# Split
train_data, test_data = train_test_split(
    dataset, test_size=0.2, random_state=42
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Train: {len(train_data)} | Test: {len(test_data)}")

# Calculate per-target pos_weights for class imbalance
pos_weights = []
for i, t in enumerate(targets):
    labels = [g.y[i].item() for g in train_data if g.y[i].item() != -1]
    neg = sum(1 for l in labels if l == 0)
    pos = sum(1 for l in labels if l == 1)
    weight = neg / pos if pos > 0 else 1.0
    pos_weights.append(weight)
    print(f"{t:<20} pos_weight: {weight:.1f}")

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float)

Train: 6243 | Test: 1561
NR-AR                pos_weight: 21.9
NR-AR-LBD            pos_weight: 26.4
NR-AhR               pos_weight: 7.8
NR-Aromatase         pos_weight: 18.8
NR-ER                pos_weight: 6.7
NR-ER-LBD            pos_weight: 18.1
NR-PPAR-gamma        pos_weight: 33.1
SR-ARE               pos_weight: 5.1
SR-ATAD5             pos_weight: 25.9
SR-HSE               pos_weight: 16.2
SR-MMP               pos_weight: 5.5
SR-p53               pos_weight: 14.6


In [9]:
from torch_geometric.nn import GATConv, global_mean_pool

class MultiTaskGNN(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=128, num_tasks=12, heads=4):
        super().__init__()
        
        self.conv1 = GATConv(input_dim, hidden_dim, heads=heads, dropout=0.2)
        self.conv2 = GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=0.2)
        self.conv3 = GATConv(hidden_dim * heads, hidden_dim, heads=1, dropout=0.2)
        
        self.shared = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        
        self.task_heads = nn.ModuleList([
            nn.Linear(64, 1) for _ in range(num_tasks)
        ])

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        x = torch.relu(self.conv1(x, edge_index))
        x = torch.dropout(x, p=0.2, train=self.training)
        x = torch.relu(self.conv2(x, edge_index))
        x = torch.dropout(x, p=0.2, train=self.training)
        x = torch.relu(self.conv3(x, edge_index))
        
        x = global_mean_pool(x, batch)
        x = self.shared(x)
        
        return torch.cat([head(x) for head in self.task_heads], dim=1)

model = MultiTaskGNN(input_dim=6, hidden_dim=128, num_tasks=len(targets), heads=4)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

MultiTaskGNN(
  (conv1): GATConv(6, 128, heads=4)
  (conv2): GATConv(512, 128, heads=4)
  (conv3): GATConv(512, 128, heads=1)
  (shared): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (task_heads): ModuleList(
    (0-11): 12 x Linear(in_features=64, out_features=1, bias=True)
  )
)

Total parameters: 343,244


In [11]:
def masked_loss(predictions, targets, pos_weights):
    # Reshape targets from [batch*12] to [batch, 12] if needed
    if targets.dim() == 1:
        targets = targets.view(-1, predictions.shape[1])
    
    total_loss = 0
    count = 0
    
    for i in range(targets.shape[1]):
        target_col = targets[:, i]
        pred_col = predictions[:, i]
        
        mask = target_col != -1
        if mask.sum() == 0:
            continue
        
        masked_pred = pred_col[mask]
        masked_target = target_col[mask]
        
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weights[i].unsqueeze(0)
        )
        total_loss += criterion(masked_pred, masked_target)
        count += 1
    
    return total_loss / count if count > 0 else torch.tensor(0.0)


def evaluate_multitask(loader):
    model.eval()
    all_preds = [[] for _ in range(len(targets))]
    all_labels = [[] for _ in range(len(targets))]
    
    with torch.no_grad():
        for batch in loader:
            out = torch.sigmoid(model(batch))
            y = batch.y.view(-1, len(targets))
            for i in range(len(targets)):
                for j in range(y.shape[0]):
                    label = y[j, i].item()
                    if label != -1:
                        all_preds[i].append(out[j, i].item())
                        all_labels[i].append(label)
    
    aucs = []
    for i, t in enumerate(targets):
        if len(set(all_labels[i])) > 1:
            auc = roc_auc_score(all_labels[i], all_preds[i])
            aucs.append(auc)
        else:
            aucs.append(None)
    return aucs

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training with early stopping
best_mean_auc = 0
patience = 10
no_improve = 0
best_state = None

print("Training multi-task GNN...\n")
for epoch in range(1, 101):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = masked_loss(out, batch.y, pos_weight_tensor)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0:
        aucs = evaluate_multitask(test_loader)
        valid_aucs = [a for a in aucs if a is not None]
        mean_auc = np.mean(valid_aucs)
        
        if mean_auc > best_mean_auc:
            best_mean_auc = mean_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f} ✓")
        else:
            no_improve += 1
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f}")
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break

print(f"\nBest Mean AUC across all 12 targets: {best_mean_auc:.4f}")


Training multi-task GNN...

Epoch   5 | Loss: 1.1389 | Mean AUC: 0.7045 ✓
Epoch  10 | Loss: 1.1024 | Mean AUC: 0.7236 ✓
Epoch  15 | Loss: 1.0809 | Mean AUC: 0.7453 ✓
Epoch  20 | Loss: 1.0490 | Mean AUC: 0.7487 ✓
Epoch  25 | Loss: 1.0357 | Mean AUC: 0.7579 ✓
Epoch  30 | Loss: 1.0271 | Mean AUC: 0.7648 ✓
Epoch  35 | Loss: 1.0018 | Mean AUC: 0.7682 ✓
Epoch  40 | Loss: 0.9926 | Mean AUC: 0.7774 ✓
Epoch  45 | Loss: 0.9892 | Mean AUC: 0.7835 ✓
Epoch  50 | Loss: 0.9654 | Mean AUC: 0.7848 ✓
Epoch  55 | Loss: 0.9636 | Mean AUC: 0.7886 ✓
Epoch  60 | Loss: 0.9441 | Mean AUC: 0.7933 ✓
Epoch  65 | Loss: 0.9343 | Mean AUC: 0.7907
Epoch  70 | Loss: 0.9121 | Mean AUC: 0.7960 ✓
Epoch  75 | Loss: 0.9194 | Mean AUC: 0.7988 ✓
Epoch  80 | Loss: 0.9003 | Mean AUC: 0.8027 ✓
Epoch  85 | Loss: 0.8900 | Mean AUC: 0.8006
Epoch  90 | Loss: 0.8837 | Mean AUC: 0.8045 ✓
Epoch  95 | Loss: 0.8693 | Mean AUC: 0.8073 ✓
Epoch 100 | Loss: 0.8639 | Mean AUC: 0.8069

Best Mean AUC across all 12 targets: 0.8073


In [12]:
print("Continuing training...\n")
for epoch in range(101, 201):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = masked_loss(out, batch.y, pos_weight_tensor)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0:
        aucs = evaluate_multitask(test_loader)
        valid_aucs = [a for a in aucs if a is not None]
        mean_auc = np.mean(valid_aucs)
        
        if mean_auc > best_mean_auc:
            best_mean_auc = mean_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f} ✓")
        else:
            no_improve += 1
            print(f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | Mean AUC: {mean_auc:.4f}")
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break

print(f"\nBest Mean AUC across all 12 targets: {best_mean_auc:.4f}")

Continuing training...

Epoch 105 | Loss: 0.8525 | Mean AUC: 0.8060
Epoch 110 | Loss: 0.8400 | Mean AUC: 0.8156 ✓
Epoch 115 | Loss: 0.8333 | Mean AUC: 0.8005
Epoch 120 | Loss: 0.8264 | Mean AUC: 0.8093
Epoch 125 | Loss: 0.8198 | Mean AUC: 0.8213 ✓
Epoch 130 | Loss: 0.8022 | Mean AUC: 0.8190
Epoch 135 | Loss: 0.7931 | Mean AUC: 0.8184
Epoch 140 | Loss: 0.7979 | Mean AUC: 0.8119
Epoch 145 | Loss: 0.7718 | Mean AUC: 0.8152
Epoch 150 | Loss: 0.8023 | Mean AUC: 0.8147
Epoch 155 | Loss: 0.7659 | Mean AUC: 0.8217 ✓
Epoch 160 | Loss: 0.7631 | Mean AUC: 0.8227 ✓
Epoch 165 | Loss: 0.7543 | Mean AUC: 0.8219
Epoch 170 | Loss: 0.7488 | Mean AUC: 0.8248 ✓
Epoch 175 | Loss: 0.7555 | Mean AUC: 0.8227
Epoch 180 | Loss: 0.7330 | Mean AUC: 0.8218
Epoch 185 | Loss: 0.7191 | Mean AUC: 0.8245
Epoch 190 | Loss: 0.7331 | Mean AUC: 0.8235
Epoch 195 | Loss: 0.7190 | Mean AUC: 0.8284 ✓
Epoch 200 | Loss: 0.7244 | Mean AUC: 0.8218

Best Mean AUC across all 12 targets: 0.8284


In [13]:
# Load best weights
model.load_state_dict(best_state)

# Per-target AUC breakdown
aucs = evaluate_multitask(test_loader)
print("Per-target AUC-ROC on test set:\n")
print(f"{'Target':<20} {'AUC':>6}")
print("-" * 28)
for t, auc in zip(targets, aucs):
    if auc is not None:
        print(f"{t:<20} {auc:.4f}")
    else:
        print(f"{t:<20}    N/A")

valid_aucs = [a for a in aucs if a is not None]
print("-" * 28)
print(f"{'Mean AUC':<20} {sum(valid_aucs)/len(valid_aucs):.4f}")

# Save model
torch.save(best_state, "best_multitask_gnn.pt")
print("\nModel saved!")

Per-target AUC-ROC on test set:

Target                  AUC
----------------------------
NR-AR                0.7739
NR-AR-LBD            0.8914
NR-AhR               0.8587
NR-Aromatase         0.8542
NR-ER                0.6714
NR-ER-LBD            0.7355
NR-PPAR-gamma        0.8824
SR-ARE               0.8242
SR-ATAD5             0.8695
SR-HSE               0.8212
SR-MMP               0.9108
SR-p53               0.8475
----------------------------
Mean AUC             0.8284

Model saved!


In [15]:
import json

results = {t: round(auc, 4) if auc else None 
           for t, auc in zip(targets, aucs)}
results["mean_auc"] = round(sum(valid_aucs)/len(valid_aucs), 4)

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved!")

Results saved!
